- Measure accuracy for devices
- Home settings

In [16]:
from __future__ import annotations
import pandas as pd 
from typing import Dict, List, Tuple
from pathlib import Path
from collections.abc import Iterable
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import numpy as np
from sklearn.preprocessing import StandardScaler
import copy
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering
import json


In [17]:
DATASET_FOLDER = "../../../dataset/processed-data"
DEVICE_MAPPING_FOLDER = DATASET_FOLDER + "/device-mapping"
INPUT_FOLDER = DATASET_FOLDER + "/input-data"
DATASET1 = "dataset1"
DATASET2 = "dataset2"
DATASET3 = "dataset3"

DATASET_FOLDER = "../../../dataset/processed-data"
DEVICE_MAPPING_FOLDER = DATASET_FOLDER + "/device-mapping"
DATASET1 = "dataset1"
DATASET2 = "dataset2"
DATASET3 = "dataset3"


DEVICE_TO_BE_INCLUDE_1 = f"{DATASET_FOLDER}/device-mapping/{DATASET1}/device_to_be_include.json"
DEVICE_TO_BE_INCLUDE_2 = f"{DATASET_FOLDER}/device-mapping/{DATASET2}/device_to_be_include.json"
DEVICE_TO_BE_INCLUDE_3 = f"{DATASET_FOLDER}/device-mapping/{DATASET3}/device_to_be_include.json"  

ALL_DATA = f"{DATASET_FOLDER}/input-data/{DATASET1}"


In [18]:
# csv_files_1 = [f"{INPUT_1}/{f.name}" for f in sorted(Path(INPUT_1).iterdir()) if f.is_file() and f.suffix == ".csv"]
# for file in csv_files_1:
#     print(file)

In [19]:
MAPPING_LIST_1 = f"{DEVICE_MAPPING_FOLDER}/{DATASET1}/list_of_devices_processed.csv"
device_mapping_1 = pd.read_csv(MAPPING_LIST_1)
device_dict_1 = (
    device_mapping_1
    .set_index("mac_address")
    .drop(columns=["mac_address"], errors="ignore")
    .to_dict(orient="index")
)

In [20]:
# device_dict_1 = {}
# with open(DEVICE_TO_BE_INCLUDE_2, "r") as f:
#     device_dict_1 = json.load(f)

In [21]:
device_dict_1

{'d0:52:a8:00:67:5e': {'device_name': 'Smart Things',
  'ip_address': nan,
  'is_iot_device': 1,
  'device_type': 'hub',
  'vendor': 'Samsung SmartThings'},
 '44:65:0d:56:cc:d3': {'device_name': 'Amazon Echo',
  'ip_address': nan,
  'is_iot_device': 1,
  'device_type': 'smart speaker',
  'vendor': 'Amazon'},
 '70:ee:50:18:34:43': {'device_name': 'Netatmo Welcome',
  'ip_address': nan,
  'is_iot_device': 1,
  'device_type': 'camera',
  'vendor': 'Netatmo'},
 'f4:f2:6d:93:51:f1': {'device_name': 'TP-Link Day Night Cloud camera',
  'ip_address': nan,
  'is_iot_device': 1,
  'device_type': 'camera',
  'vendor': 'TP-Link'},
 '00:16:6c:ab:6b:88': {'device_name': 'Samsung SmartCam',
  'ip_address': nan,
  'is_iot_device': 1,
  'device_type': 'camera',
  'vendor': 'Samsung'},
 '30:8c:fb:2f:e4:b2': {'device_name': 'Dropcam',
  'ip_address': nan,
  'is_iot_device': 1,
  'device_type': 'camera',
  'vendor': 'Dropcam'},
 '00:62:6e:51:27:2e': {'device_name': 'Insteon Camera Wired',
  'ip_address': 

In [22]:
FEATURE_COLUMNS = [
    "packet_count",
    "total_bytes",
]

DEVICE_COLUMN = "eth.src"          # Replace with "eth.src" if applicable
TIME_COLUMN = "timestamp"
LABEL_COLUMN = "is_iot_device"

KPLITS = 5
RANDOM_STATE = 42
WINDOW_SECONDS = 4

In [23]:
# Outgoing volume
def extract_data_from_pcap(
    file_paths: Iterable[str]
) -> pd.DataFrame:
    columns = [
        "frame.number",
        "frame.time_epoch",
        "frame.len",
        "eth.src",
        "eth.dst",
    ]

    all_data = []

    for file_path in file_paths:
        raw_df = pd.read_csv(
            file_path,
            usecols=columns,
            dtype={
                "frame.number": "int32",
                "frame.time_epoch": "float64",
                "frame.len": "int32",
                "eth.src": "category",
                "eth.dst": "category",
            }
        )

        raw_df["capture_id"] = Path(file_path).stem
        all_data.append(raw_df)


    res = pd.concat(all_data, ignore_index=True)
    res = res.sort_values(
        ["frame.time_epoch", "eth.src"],
        ignore_index=True,
    )

    return res


In [24]:
GATEWAY_MAC = "14:cc:20:51:33:ea"

In [25]:
# all_data = extract_data_from_pcap(csv_files_1)
df = pd.read_parquet(
    ALL_DATA + "/all_data.parquet",
    columns=[
        "frame.len",
        "eth.src",
        "eth.dst",
    ]
)

In [26]:
df.head(5)

,frame.len,eth.src,eth.dst
0,70,18:b7:9e:02:20:44,14:cc:20:51:33:ea
1,66,18:b7:9e:02:20:44,14:cc:20:51:33:ea
2,66,14:cc:20:51:33:ea,30:8c:fb:2f:e4:b2
3,60,d0:52:a8:00:67:5e,14:cc:20:51:33:ea
4,98,14:cc:20:51:33:ea,70:ee:50:18:34:43


- A lot of traffic that are Ipv6 multicast neighbor discovery like 33:33:ff:xx:xx:xx. We'll classify if a mac is broadcast, unicast, or multicast
- If the mac are all FF -> broadcast
- Least significant bit of the first octect = 0 -> unicast
- Least significant bit of the first octect = 1 -> multicast

In [27]:
def classify_mac(mac):
    if pd.isna(mac):
        return "invalid"
    
    if mac in device_dict_1:
        return "unicast"
    
    mac = str(mac).lower().replace("-", ":")
    parts = mac.split(":")

    if len(parts) != 6:
        return "invalid"

    try:
        octets = [int(x, 16) for x in parts]
    except ValueError:
        return "invalid"

    if octets == [0xff] * 6:
        return "broadcast"

    if octets[0] & 0x01:
        return "multicast"

    return "unicast"

In [28]:
def compute_incoming_and_outgoing_data(df: pd.DataFrame, device_dict_1: dict) -> dict:
    device_info = {}

    for mac in device_dict_1:

        source_mac = mac

        # ============================================================
        # Outgoing traffic
        # source_mac -> destination
        # ============================================================

        outgoing_df = (
            df[df["eth.src"] == source_mac]
            .dropna(subset=["eth.dst"])
            .groupby("eth.dst", observed=True)
            .agg(
                outgoing_packet_count=("frame.len", "size"),
                outgoing_total_bytes=("frame.len", "sum")
            )
            .reset_index()
            .rename(columns={"eth.dst": "peer_mac"})
            .sort_values("outgoing_packet_count", ascending=False)
        )

        outgoing_df["peer_type"] = outgoing_df["peer_mac"].apply(classify_mac)

        # Drop unknown unicast destinations
        outgoing_df = outgoing_df[
            ~(
                (outgoing_df["peer_type"] == "unicast")
                & (~outgoing_df["peer_mac"].isin(device_dict_1.keys()))
            )
        ].copy()

        # Percentage of total outgoing packets/bytes
        outgoing_df["outgoing_packet_count_pct"] = (
            outgoing_df["outgoing_packet_count"]
            / outgoing_df["outgoing_packet_count"].sum()
            * 100
        )

        outgoing_df["outgoing_total_bytes_pct"] = (
            outgoing_df["outgoing_total_bytes"]
            / outgoing_df["outgoing_total_bytes"].sum()
            * 100
        )

        outgoing_df["direction"] = "outgoing"


        # ============================================================
        # Incoming traffic
        # source -> source_mac
        # ============================================================

        incoming_df = (
            df[df["eth.dst"] == source_mac]
            .dropna(subset=["eth.src"])
            .groupby("eth.src", observed=True)
            .agg(
                incoming_packet_count=("frame.len", "size"),
                incoming_total_bytes=("frame.len", "sum")
            )
            .reset_index()
            .rename(columns={"eth.src": "peer_mac"})
            .sort_values("incoming_packet_count", ascending=False)
        )

        incoming_df["peer_type"] = incoming_df["peer_mac"].apply(classify_mac)

        # Drop unknown unicast sources
        incoming_df = incoming_df[
            ~(
                (incoming_df["peer_type"] == "unicast")
                & (~incoming_df["peer_mac"].isin(device_dict_1.keys()))
            )
        ].copy()

        # Percentage of total incoming packets/bytes
        incoming_df["incoming_packet_count_pct"] = (
            incoming_df["incoming_packet_count"]
            / incoming_df["incoming_packet_count"].sum()
            * 100
        )

        incoming_df["incoming_total_bytes_pct"] = (
            incoming_df["incoming_total_bytes"]
            / incoming_df["incoming_total_bytes"].sum()
            * 100
        )

        incoming_df["direction"] = "incoming"


        # ============================================================
        # Store results
        # ============================================================

        print(
            f"Traffic for "
            f"{device_dict_1[source_mac]['device_name']} "
            f"({source_mac})"
        )

        device_info[source_mac] = {
            "device_name": device_dict_1[source_mac]["device_name"],
            "outgoing_traffic_summary": outgoing_df,
            "incoming_traffic_summary": incoming_df
        }

        print("\tOutgoing:")
        display(outgoing_df.sort_values("peer_type"))

        print("\tIncoming:")
        display(incoming_df.sort_values("peer_type"))
        print()
    return device_info

device_info = compute_incoming_and_outgoing_data(df=df, device_dict_1=device_dict_1)

Traffic for Smart Things (d0:52:a8:00:67:5e)
	Outgoing:


,peer_mac,outgoing_packet_count,outgoing_total_bytes,peer_type,outgoing_packet_count_pct,outgoing_total_bytes_pct,direction
2,ff:ff:ff:ff:ff:ff,6113,379188,broadcast,2.102616,1.826085,outgoing
1,14:cc:20:51:33:ea,284620,20385886,unicast,97.897384,98.173915,outgoing


	Incoming:


,peer_mac,incoming_packet_count,incoming_total_bytes,peer_type,incoming_packet_count_pct,incoming_total_bytes_pct,direction
0,14:cc:20:51:33:ea,280308,19224381,unicast,100.0,100.0,incoming



Traffic for Amazon Echo (44:65:0d:56:cc:d3)
	Outgoing:


,peer_mac,outgoing_packet_count,outgoing_total_bytes,peer_type,outgoing_packet_count_pct,outgoing_total_bytes_pct,direction
5,ff:ff:ff:ff:ff:ff,3029,129104,broadcast,0.430932,0.158794,outgoing
0,01:00:5e:00:00:16,1847,99762,multicast,0.262770,0.122705,outgoing
2,01:00:5e:7f:ff:fa,1556,241958,multicast,0.221370,0.297602,outgoing
1,01:00:5e:00:00:fb,37,10505,multicast,0.005264,0.012921,outgoing
4,14:cc:20:51:33:ea,696426,80821261,unicast,99.079663,99.407978,outgoing


... (114 outputs left out)


Traffic for TPLink Router Bridge LAN (Gateway) (14:cc:20:51:33:ea)
	Outgoing:


,peer_mac,outgoing_packet_count,outgoing_total_bytes,peer_type,outgoing_packet_count_pct,outgoing_total_bytes_pct,direction
47,ff:ff:ff:ff:ff:ff,27243,1170016,broadcast,0.221068,1.209959e-02,outgoing
15,33:33:00:00:00:0c,195685,97955690,multicast,1.587920,1.012998e+00,outgoing
7,01:00:5e:7f:ff:fa,64883,31252872,multicast,0.526505,3.231982e-01,outgoing
13,33:33:00:00:00:01,14539,1370866,multicast,0.117979,1.417666e-02,outgoing
22,33:33:ff:51:33:ea,13465,1157990,multicast,0.109264,1.197523e-02,outgoing
16,33:33:00:01:00:02,13465,1157990,multicast,0.109264,1.197523e-02,outgoing
17,33:33:00:01:00:03,13464,1157904,multicast,0.109256,1.197434e-02,outgoing
19,33:33:ff:00:00:01,13464,1157904,multicast,0.109256,1.197434e-02,outgoing
18,33:33:ff:00:00:00,13385,1151110,multicast,0.108615,1.190408e-02,outgoing
14,33:33:00:00:00:02,13373,1150078,multicast,0.108518,1.189341e-02,outgoing


	Incoming:


,peer_mac,incoming_packet_count,incoming_total_bytes,peer_type,incoming_packet_count_pct,incoming_total_bytes_pct,direction
17,74:2f:68:81:69:42,2318577,208892918,unicast,25.665445,14.231978,incoming
9,30:8c:fb:2f:e4:b2,2131288,440445331,unicast,23.592253,30.007759,incoming
12,44:65:0d:56:cc:d3,696426,80821261,unicast,7.709075,5.506393,incoming
20,ac:bc:32:d4:6f:2f,660769,87784886,unicast,7.314370,5.980828,incoming
6,08:21:ef:3b:fc:e3,430077,50686502,unicast,4.760729,3.453297,incoming
5,00:62:6e:51:27:2e,399317,36533356,unicast,4.420232,2.489036,incoming
2,00:24:e4:11:18:a8,350688,27868705,unicast,3.881934,1.898709,incoming
16,70:ee:50:18:34:43,335459,171288996,unicast,3.713357,11.670004,incoming
22,d0:52:a8:00:67:5e,284620,20385886,unicast,3.150596,1.388901,incoming
0,00:16:6c:ab:6b:88,258413,26359197,unicast,2.860498,1.795865,incoming


In [29]:
print(
    "Identifying potential routers based on unicast traffic "
    "and its corresponding traffic rates..."
)

potential_non_iot = set()

for mac in device_info:
    # if mac == GATEWAY_MAC:
    #     continue

    traffic_summary = device_info[mac]["outgoing_traffic_summary"]

    # Only consider known devices
    traffic_summary = traffic_summary[
        traffic_summary["peer_mac"].isin(device_dict_1.keys())
        & (traffic_summary["peer_mac"] != GATEWAY_MAC)
    ]

    # Only unicast destinations
    unicast_traffic = traffic_summary[
        traffic_summary["peer_type"] == "unicast"
    ]

    unicast_count = len(unicast_traffic)
    total_devices = len(device_dict_1) - 1 # Filter router

    if unicast_count / total_devices < 0.2:
        if device_dict_1[mac]["is_iot_device"] == 0:
            print(
                f"Missclassified {device_dict_1[mac]["device_name"]}: {mac}, "
                f"{unicast_count}/{total_devices} devices ({unicast_count / total_devices:.2%})\n"
                f"\tOutgoing traffic:\n"
            )
            display(device_info[mac]["outgoing_traffic_summary"].sort_values("peer_type"))
            print(f"\tIncoming traffic:\n")
            display(device_info[mac]["incoming_traffic_summary"].sort_values("peer_type"))
        continue
    potential_non_iot.add(mac)
    print(
        f"Potential non-IoT: "
        f"{device_info[mac]['device_name']} ({mac}) "
        f"- communicates with "
        f"{unicast_count}/{total_devices} devices "
        f"({unicast_count / total_devices:.2%})"
    )

correct_prediction = 0
all_device_but_gw = len(device_dict_1.keys()) - 1
for mac in device_dict_1:
    # if mac == GATEWAY_MAC:
    #     continue
    if mac in potential_non_iot and device_dict_1[mac]["is_iot_device"] == 0:
        correct_prediction += 1
    elif mac not in potential_non_iot and device_dict_1[mac]["is_iot_device"] == 1:
        correct_prediction += 1

print(f"Accuracy Rate: {correct_prediction / all_device_but_gw:.2%}")

Identifying potential routers based on unicast traffic and its corresponding traffic rates...
Potential non-IoT: Samsung SmartCam (00:16:6c:ab:6b:88) - communicates with 8/30 devices (26.67%)
Potential non-IoT: Samsung Galaxy Tab (08:21:ef:3b:fc:e3) - communicates with 11/30 devices (36.67%)
Missclassified Android Phone: 40:f3:08:ff:1e:da, 4/30 devices (13.33%)
	Outgoing traffic:



,peer_mac,outgoing_packet_count,outgoing_total_bytes,peer_type,outgoing_packet_count_pct,outgoing_total_bytes_pct,direction
14,ff:ff:ff:ff:ff:ff,12,1716,broadcast,0.205374,0.161385,outgoing
4,01:00:5e:7f:ff:fa,103,8608,multicast,1.762793,0.809560,outgoing
3,01:00:5e:00:00:fb,41,3596,multicast,0.701694,0.338194,outgoing
2,01:00:5e:00:00:16,10,540,multicast,0.171145,0.050786,outgoing
1,01:00:5e:00:00:02,7,322,multicast,0.119801,0.030283,outgoing
10,33:33:ff:ff:1e:da,7,578,multicast,0.119801,0.054359,outgoing
8,33:33:00:00:00:16,6,620,multicast,0.102687,0.058309,outgoing
9,33:33:ff:60:4a:2c,4,336,multicast,0.068458,0.031600,outgoing
7,33:33:00:00:00:02,1,70,multicast,0.017114,0.006583,outgoing
6,14:cc:20:51:33:ea,4678,958455,unicast,80.061612,90.140168,outgoing


	Incoming traffic:



,peer_mac,incoming_packet_count,incoming_total_bytes,peer_type,incoming_packet_count_pct,incoming_total_bytes_pct,direction
1,14:cc:20:51:33:ea,4144,1583199,unicast,84.090909,90.447065,incoming
2,74:2f:68:81:69:42,622,80229,unicast,12.621753,4.583427,incoming
3,ec:1a:59:79:f4:89,69,40411,unicast,1.400162,2.308653,incoming
4,ec:1a:59:83:28:11,69,40399,unicast,1.400162,2.307967,incoming
0,00:16:6c:ab:6b:88,24,6177,unicast,0.487013,0.352888,incoming


Potential non-IoT: Laptop (74:2f:68:81:69:42) - communicates with 10/30 devices (33.33%)
Potential non-IoT: MacBook (ac:bc:32:d4:6f:2f) - communicates with 6/30 devices (20.00%)
Missclassified Android Phone: b4:ce:f6:a7:a3:c2, 0/30 devices (0.00%)
	Outgoing traffic:



,peer_mac,outgoing_packet_count,outgoing_total_bytes,peer_type,outgoing_packet_count_pct,outgoing_total_bytes_pct,direction
14,ff:ff:ff:ff:ff:ff,411,42078,broadcast,1.017579,0.629081,outgoing
9,33:33:ff:a7:a3:c2,92,7648,multicast,0.227779,0.114340,outgoing
3,01:00:5e:02:00:fc,82,6866,multicast,0.203021,0.102649,outgoing
7,33:33:00:00:00:16,47,4490,multicast,0.116365,0.067127,outgoing
12,33:33:ff:cc:2a:04,26,2212,multicast,0.064372,0.033070,outgoing
6,33:33:00:00:00:02,25,1750,multicast,0.061897,0.026163,outgoing
10,33:33:ff:b2:9f:aa,22,1868,multicast,0.054469,0.027927,outgoing
2,01:00:5e:00:00:fb,21,1821,multicast,0.051993,0.027225,outgoing
8,33:33:ff:47:a0:a9,19,1594,multicast,0.047041,0.023831,outgoing
13,33:33:ff:cd:9d:9b,8,656,multicast,0.019807,0.009807,outgoing


... (7 outputs left out)

,peer_mac,outgoing_packet_count,outgoing_total_bytes,peer_type,outgoing_packet_count_pct,outgoing_total_bytes_pct,direction
8,ff:ff:ff:ff:ff:ff,97,15592,broadcast,1.007374,0.720011,outgoing
2,01:00:5e:7f:ff:fa,40,8680,multicast,0.415412,0.400827,outgoing
1,01:00:5e:00:00:fb,34,10083,multicast,0.353100,0.465615,outgoing
0,01:00:5e:00:00:02,1,46,multicast,0.010385,0.002124,outgoing
5,33:33:ff:2c:0b:11,1,86,multicast,0.010385,0.003971,outgoing
6,33:33:ff:93:cc:85,1,86,multicast,0.010385,0.003971,outgoing
4,14:cc:20:51:33:ea,9447,2130109,unicast,98.109876,98.364690,outgoing
7,70:5a:0f:e4:9b:c0,8,840,unicast,0.083082,0.038790,outgoing


	Incoming traffic:



,peer_mac,incoming_packet_count,incoming_total_bytes,peer_type,incoming_packet_count_pct,incoming_total_bytes_pct,direction
0,14:cc:20:51:33:ea,8874,4053839,unicast,99.90993,99.95717,incoming
1,70:5a:0f:e4:9b:c0,8,1737,unicast,0.09007,0.04283,incoming


Potential non-IoT: TPLink Router Bridge LAN (Gateway) (14:cc:20:51:33:ea) - communicates with 30/30 devices (100.00%)
Accuracy Rate: 86.67%


In [30]:
print(f"Actual non-IoT devices: ")
for mac in device_dict_1:
    if device_dict_1[mac]["is_iot_device"] == 0 and mac != GATEWAY_MAC:
        print(f"\t{device_dict_1[mac]["device_name"]}: {mac}")

Actual non-IoT devices: 
	Samsung Galaxy Tab: 08:21:ef:3b:fc:e3
	Android Phone: 40:f3:08:ff:1e:da
	Laptop: 74:2f:68:81:69:42
	MacBook: ac:bc:32:d4:6f:2f
	Android Phone: b4:ce:f6:a7:a3:c2
	IPhone: d0:a6:37:df:a1:e1
	MacBook/Iphone: f4:5c:89:93:cc:85


# Identify Router

In [ ]:
# unicast_summary = []

# for mac, info in device_info.items():

#     outgoing_df = info["outgoing_traffic_summary"]
#     incoming_df = info["incoming_traffic_summary"]

#     outgoing_unicast = outgoing_df[
#         outgoing_df["peer_type"] == "unicast"
#     ]

#     incoming_unicast = incoming_df[
#         incoming_df["peer_type"] == "unicast"
#     ]

#     outgoing_packets = outgoing_unicast["outgoing_packet_count"].sum()
#     outgoing_bytes = outgoing_unicast["outgoing_total_bytes"].sum()

#     incoming_packets = incoming_unicast["incoming_packet_count"].sum()
#     incoming_bytes = incoming_unicast["incoming_total_bytes"].sum()

#     outgoing_multicast = outgoing_df[
#         outgoing_df["peer_type"] == "multicast"
#     ]

#     outgoing_multicast_distinct_conversation = outgoing_multicast["peer_mac"].nunique()
#     outgoing_multicast_packets = outgoing_multicast["outgoing_packet_count"].sum()
#     outgoing_multicast_bytes = outgoing_multicast["outgoing_total_bytes"].sum()

#     unicast_summary.append({
#         "mac": mac,
#         "device_name": info["device_name"],
#         "outgoing_packet_count": outgoing_packets,
#         "outgoing_total_bytes": outgoing_bytes,
#         "incoming_packet_count": incoming_packets,
#         "incoming_total_bytes": incoming_bytes,
#         "total_packet_count": outgoing_packets + incoming_packets,
#         "total_bytes": outgoing_bytes + incoming_bytes,
#         "distinct_multicast_conversations": outgoing_multicast_distinct_conversation,
#         "outgoing_multicast_packet_count": outgoing_multicast_packets,
#         "outgoing_multicast_total_bytes": outgoing_multicast_bytes
#     })

# unicast_summary_df = pd.DataFrame(unicast_summary)

# display(unicast_summary_df.sort_values(by=["total_packet_count", "total_bytes"], ascending=False))

In [ ]:

# feature_cols = ["outgoing_packet_count","outgoing_total_bytes","incoming_packet_count","incoming_total_bytes", "total_packet_count", "total_bytes","distinct_multicast_conversations","outgoing_multicast_packet_count","outgoing_multicast_total_bytes"]
# # feature_cols = ["total_packet_count", "total_bytes","distinct_multicast_conversations","outgoing_multicast_total_bytes"]
# features = unicast_summary_df[
#     feature_cols
# ]

# # features = unicast_summary_df[
# #     ["total_packet_count"]
# # ]

# # X = np.log1p(features)
# scaler = StandardScaler()
# X_scaled = scaler.fit_transform(features)

# # kmeans = KMeans(
# #     n_clusters=2,
# #     random_state=42,
# #     n_init=20
# # )
# # unicast_summary_df["cluster"] = kmeans.fit_predict(X_scaled)

# # dbscan = DBSCAN(
# #     eps=1.5,
# #     min_samples=3
# # )

# # unicast_summary_df["cluster"] = dbscan.fit_predict(X_scaled)

# model = AgglomerativeClustering(
#     n_clusters=2
# )

# unicast_summary_df["cluster"] = (
#     model.fit_predict(X_scaled)
# )

# display(
#     unicast_summary_df[
#         [
#             "mac",
#             "device_name",
#             "cluster"
#         ] + feature_cols
#     ].sort_values("cluster")
# )

In [ ]:
# from itertools import combinations

# import pandas as pd
# import matplotlib.pyplot as plt


# def plotKMeans(
#     unicast_summary_df: pd.DataFrame,
#     feature_columns: list[str],
# ) -> None:

#     if len(feature_columns) < 2:
#         raise ValueError("feature_columns must contain at least 2 features.")

#     # Validate columns
#     required_columns = set(feature_columns) | {"cluster", "device_name"}
#     missing_columns = required_columns - set(unicast_summary_df.columns)

#     if missing_columns:
#         raise ValueError(
#             f"Missing columns in dataframe: {missing_columns}"
#         )

#     # Generate every pair of features
#     feature_pairs = combinations(feature_columns, 2)

#     for x_feature, y_feature in feature_pairs:

#         plt.figure(figsize=(10, 7))

#         # Plot each cluster
#         for cluster in sorted(unicast_summary_df["cluster"].unique()):

#             cluster_df = unicast_summary_df[
#                 unicast_summary_df["cluster"] == cluster
#             ]

#             plt.scatter(
#                 cluster_df[x_feature],
#                 cluster_df[y_feature],
#                 s=70,
#                 label=f"Cluster {cluster}"
#             )

#         # Label every device
#         for _, row in unicast_summary_df.iterrows():
#             plt.annotate(
#                 row["device_name"],
#                 (
#                     row[x_feature],
#                     row[y_feature]
#                 ),
#                 xytext=(5, 5),
#                 textcoords="offset points",
#                 fontsize=7
#             )

#         # Use log scale
#         plt.xscale("log")
#         plt.yscale("log")

#         plt.xlabel(x_feature)
#         plt.ylabel(y_feature)

#         plt.title(
#             f"Device Clustering: {x_feature} vs {y_feature}"
#         )

#         plt.legend()
#         plt.grid(alpha=0.3)
#         plt.tight_layout()
#         plt.savefig(
#             f"imgs/device_clustering_{x_feature}_vs_{y_feature}.png",
#             dpi=300,
#             bbox_inches="tight"
#         )
#         plt.show()
# plotKMeans(unicast_summary_df, feature_cols)

## Generate Mock Router

In [ ]:
# def create_mock_gateway_summary(
#     device_info,
#     unicast_summary_df,
#     device_dict,
#     gateway_mac,
#     n_gateways=3,
#     random_state=42,
# ):
#     if n_gateways < 1:
#         raise ValueError("n_gateways must be >= 1")

#     rng = np.random.default_rng(random_state)

#     # ============================================================
#     # Create synthetic gateway MACs
#     # ============================================================

#     mock_gateways = [
#         f"02:00:00:00:00:{i + 1:02x}"
#         for i in range(n_gateways)
#     ]

#     main_gateway = rng.choice(mock_gateways)

#     # ============================================================
#     # Randomly assign devices to gateways
#     # ============================================================

#     endpoint_devices = [
#         mac
#         for mac in device_dict
#         if mac != gateway_mac
#     ]

#     shuffled = endpoint_devices.copy()
#     rng.shuffle(shuffled)

#     device_to_gateway = {
#         mac: mock_gateways[i % n_gateways]
#         for i, mac in enumerate(shuffled)
#     }

#     # ============================================================
#     # Get original gateway <-> device traffic
#     # ============================================================

#     gateway_info = device_info[gateway_mac]

#     outgoing = gateway_info[
#         "outgoing_traffic_summary"
#     ].copy()

#     incoming = gateway_info[
#         "incoming_traffic_summary"
#     ].copy()

#     # Keep known unicast devices only
#     outgoing = outgoing[
#         (outgoing["peer_type"] == "unicast")
#         & outgoing["peer_mac"].isin(endpoint_devices)
#     ].copy()

#     incoming = incoming[
#         (incoming["peer_type"] == "unicast")
#         & incoming["peer_mac"].isin(endpoint_devices)
#     ].copy()

#     # ============================================================
#     # Make one row per peer
#     # ============================================================

#     outgoing = outgoing[
#         [
#             "peer_mac",
#             "outgoing_packet_count",
#             "outgoing_total_bytes",
#         ]
#     ]

#     incoming = incoming[
#         [
#             "peer_mac",
#             "incoming_packet_count",
#             "incoming_total_bytes",
#         ]
#     ]

#     peer_traffic = pd.merge(
#         outgoing,
#         incoming,
#         on="peer_mac",
#         how="outer",
#     ).fillna(0)

#     peer_traffic["assigned_gateway"] = (
#         peer_traffic["peer_mac"]
#         .map(device_to_gateway)
#     )

#     # ============================================================
#     # Aggregate traffic assigned to each gateway
#     # ============================================================

#     gateway_load = (
#         peer_traffic
#         .groupby("assigned_gateway", as_index=False)
#         .agg(
#             original_outgoing_packets=(
#                 "outgoing_packet_count",
#                 "sum"
#             ),
#             original_outgoing_bytes=(
#                 "outgoing_total_bytes",
#                 "sum"
#             ),
#             original_incoming_packets=(
#                 "incoming_packet_count",
#                 "sum"
#             ),
#             original_incoming_bytes=(
#                 "incoming_total_bytes",
#                 "sum"
#             ),
#         )
#         .set_index("assigned_gateway")
#         .reindex(mock_gateways, fill_value=0)
#     )

#     # ============================================================
#     # Construct synthetic gateway totals
#     # ============================================================

#     gateway_rows = []

#     # Original gateway directional totals
#     original_total_out_packets = (
#         peer_traffic["outgoing_packet_count"].sum()
#     )

#     original_total_out_bytes = (
#         peer_traffic["outgoing_total_bytes"].sum()
#     )

#     original_total_in_packets = (
#         peer_traffic["incoming_packet_count"].sum()
#     )

#     original_total_in_bytes = (
#         peer_traffic["incoming_total_bytes"].sum()
#     )

#     for i, gw in enumerate(mock_gateways):

#         load = gateway_load.loc[gw]

#         if gw == main_gateway:

#             # Main gateway sees every original gateway packet once
#             outgoing_packets = original_total_out_packets
#             outgoing_bytes = original_total_out_bytes

#             incoming_packets = original_total_in_packets
#             incoming_bytes = original_total_in_bytes

#             name = f"Mock Gateway {i + 1} (Main)"

#         else:

#             # Downstream gateway participates on BOTH links:
#             #
#             # device <-> downstream GW
#             # downstream GW <-> main GW
#             #
#             # Every original packet therefore contributes
#             # one incoming + one outgoing packet.

#             traffic_packets = (
#                 load["original_outgoing_packets"]
#                 + load["original_incoming_packets"]
#             )

#             traffic_bytes = (
#                 load["original_outgoing_bytes"]
#                 + load["original_incoming_bytes"]
#             )

#             outgoing_packets = traffic_packets
#             incoming_packets = traffic_packets

#             outgoing_bytes = traffic_bytes
#             incoming_bytes = traffic_bytes

#             name = f"Mock Gateway {i + 1}"

#         gateway_rows.append({
#             "mac": gw,
#             "device_name": name,

#             "outgoing_packet_count": outgoing_packets,
#             "outgoing_total_bytes": outgoing_bytes,

#             "incoming_packet_count": incoming_packets,
#             "incoming_total_bytes": incoming_bytes,

#             "total_packet_count": (
#                 outgoing_packets
#                 + incoming_packets
#             ),

#             "total_bytes": (
#                 outgoing_bytes
#                 + incoming_bytes
#             ),

#             "is_mock_gateway": True,
#             "is_main_gateway": gw == main_gateway,
#         })

#     gateway_summary_df = pd.DataFrame(gateway_rows)

#     # ============================================================
#     # Keep original endpoint totals
#     # ============================================================

#     device_summary = unicast_summary_df[
#         unicast_summary_df["mac"] != gateway_mac
#     ].copy()

#     device_summary["is_mock_gateway"] = False
#     device_summary["is_main_gateway"] = False

#     # ============================================================
#     # Combine endpoints + synthetic gateways
#     # ============================================================

#     result_df = pd.concat(
#         [
#             device_summary,
#             gateway_summary_df,
#         ],
#         ignore_index=True,
#         sort=False,
#     )

#     # ============================================================
#     # Assignment table
#     # ============================================================

#     assignment_df = pd.DataFrame([
#         {
#             "device_mac": mac,
#             "device_name": device_dict[mac]["device_name"],
#             "assigned_gateway": gw,
#             "assigned_to_main": gw == main_gateway,
#         }
#         for mac, gw in device_to_gateway.items()
#     ])

#     return (
#         result_df,
#         gateway_summary_df,
#         assignment_df,
#         main_gateway,
#         peer_traffic,
#     )

In [ ]:
# def create_mock_gateway_summary(
#     device_info,
#     unicast_summary_df,
#     device_dict,
#     gateway_mac,
#     n_gateways=3,
#     random_state=42,
# ):
#     if n_gateways < 1:
#         raise ValueError("n_gateways must be >= 1")

#     rng = np.random.default_rng(random_state)

#     # ============================================================
#     # Create synthetic gateway MACs
#     # ============================================================

#     mock_gateways = [
#         f"02:00:00:00:00:{i + 1:02x}"
#         for i in range(n_gateways)
#     ]

#     main_gateway = rng.choice(mock_gateways)

#     # ============================================================
#     # Randomly assign devices to gateways
#     # ============================================================

#     endpoint_devices = [
#         mac
#         for mac in device_dict
#         if mac != gateway_mac
#     ]

#     shuffled = endpoint_devices.copy()
#     rng.shuffle(shuffled)

#     device_to_gateway = {
#         mac: mock_gateways[i % n_gateways]
#         for i, mac in enumerate(shuffled)
#     }

#     # ============================================================
#     # Get original gateway traffic
#     # ============================================================

#     gateway_info = device_info[gateway_mac]

#     outgoing = gateway_info[
#         "outgoing_traffic_summary"
#     ].copy()

#     incoming = gateway_info[
#         "incoming_traffic_summary"
#     ].copy()

#     # ============================================================
#     # Extract original gateway multicast traffic
#     #
#     # Assumption:
#     # Every mock gateway participates in the same multicast
#     # conversations and therefore gets identical multicast features.
#     # ============================================================

#     outgoing_multicast = outgoing[
#         outgoing["peer_type"] == "multicast"
#     ].copy()

#     original_multicast_conversations = (
#         outgoing_multicast["peer_mac"].nunique()
#     )

#     original_multicast_packets = (
#         outgoing_multicast["outgoing_packet_count"].sum()
#     )

#     original_multicast_bytes = (
#         outgoing_multicast["outgoing_total_bytes"].sum()
#     )

#     # ============================================================
#     # Keep known unicast devices only
#     # ============================================================

#     outgoing_unicast = outgoing[
#         (outgoing["peer_type"] == "unicast")
#         & outgoing["peer_mac"].isin(endpoint_devices)
#     ].copy()

#     incoming_unicast = incoming[
#         (incoming["peer_type"] == "unicast")
#         & incoming["peer_mac"].isin(endpoint_devices)
#     ].copy()

#     # ============================================================
#     # Keep only columns needed for peer-level traffic
#     # ============================================================

#     outgoing_unicast = outgoing_unicast[
#         [
#             "peer_mac",
#             "outgoing_packet_count",
#             "outgoing_total_bytes",
#         ]
#     ]

#     incoming_unicast = incoming_unicast[
#         [
#             "peer_mac",
#             "incoming_packet_count",
#             "incoming_total_bytes",
#         ]
#     ]

#     # ============================================================
#     # Merge outgoing and incoming traffic
#     #
#     # One row per peer device
#     # ============================================================

#     peer_traffic = pd.merge(
#         outgoing_unicast,
#         incoming_unicast,
#         on="peer_mac",
#         how="outer",
#     ).fillna(0)

#     # Assign each peer device to a mock gateway
#     peer_traffic["assigned_gateway"] = (
#         peer_traffic["peer_mac"]
#         .map(device_to_gateway)
#     )

#     # ============================================================
#     # Aggregate original gateway <-> device traffic
#     # for each mock gateway
#     # ============================================================

#     gateway_load = (
#         peer_traffic
#         .groupby(
#             "assigned_gateway",
#             as_index=False,
#         )
#         .agg(
#             original_outgoing_packets=(
#                 "outgoing_packet_count",
#                 "sum",
#             ),
#             original_outgoing_bytes=(
#                 "outgoing_total_bytes",
#                 "sum",
#             ),
#             original_incoming_packets=(
#                 "incoming_packet_count",
#                 "sum",
#             ),
#             original_incoming_bytes=(
#                 "incoming_total_bytes",
#                 "sum",
#             ),
#         )
#         .set_index("assigned_gateway")
#         .reindex(
#             mock_gateways,
#             fill_value=0,
#         )
#     )

#     # ============================================================
#     # Original gateway directional UNICAST totals
#     # ============================================================

#     original_total_out_packets = (
#         peer_traffic["outgoing_packet_count"].sum()
#     )

#     original_total_out_bytes = (
#         peer_traffic["outgoing_total_bytes"].sum()
#     )

#     original_total_in_packets = (
#         peer_traffic["incoming_packet_count"].sum()
#     )

#     original_total_in_bytes = (
#         peer_traffic["incoming_total_bytes"].sum()
#     )

#     # ============================================================
#     # Construct synthetic gateway summaries
#     # ============================================================

#     gateway_rows = []

#     for i, gw in enumerate(mock_gateways):

#         load = gateway_load.loc[gw]

#         # ========================================================
#         # Main gateway
#         # ========================================================

#         if gw == main_gateway:

#             # Main gateway sees all original gateway traffic
#             # because downstream gateway traffic eventually
#             # reaches the main gateway.
#             outgoing_packets = (
#                 original_total_out_packets
#             )

#             outgoing_bytes = (
#                 original_total_out_bytes
#             )

#             incoming_packets = (
#                 original_total_in_packets
#             )

#             incoming_bytes = (
#                 original_total_in_bytes
#             )

#             name = f"Mock Gateway {i + 1} (Main)"

#         # ========================================================
#         # Downstream gateway
#         # ========================================================

#         else:

#             # A downstream gateway participates in two links:
#             #
#             # device <-> downstream gateway
#             #
#             # downstream gateway <-> main gateway
#             #
#             # Therefore, every original packet associated with
#             # this downstream gateway contributes one incoming
#             # and one outgoing packet from the downstream
#             # gateway's perspective.

#             traffic_packets = (
#                 load["original_outgoing_packets"]
#                 + load["original_incoming_packets"]
#             )

#             traffic_bytes = (
#                 load["original_outgoing_bytes"]
#                 + load["original_incoming_bytes"]
#             )

#             outgoing_packets = traffic_packets
#             incoming_packets = traffic_packets

#             outgoing_bytes = traffic_bytes
#             incoming_bytes = traffic_bytes

#             name = f"Mock Gateway {i + 1}"

#         # ========================================================
#         # Multicast features
#         #
#         # Replicate original gateway multicast behavior across
#         # ALL mock gateways.
#         # ========================================================

#         distinct_multicast_conversations = (
#             original_multicast_conversations
#         )

#         outgoing_multicast_packet_count = (
#             original_multicast_packets
#         )

#         outgoing_multicast_total_bytes = (
#             original_multicast_bytes
#         )

#         # ========================================================
#         # Create gateway summary row
#         # ========================================================

#         gateway_rows.append({
#             "mac": gw,

#             "device_name": name,

#             # ----------------------------
#             # Unicast features
#             # ----------------------------

#             "outgoing_packet_count": (
#                 outgoing_packets
#             ),

#             "outgoing_total_bytes": (
#                 outgoing_bytes
#             ),

#             "incoming_packet_count": (
#                 incoming_packets
#             ),

#             "incoming_total_bytes": (
#                 incoming_bytes
#             ),

#             "total_packet_count": (
#                 outgoing_packets
#                 + incoming_packets
#             ),

#             "total_bytes": (
#                 outgoing_bytes
#                 + incoming_bytes
#             ),

#             # ----------------------------
#             # Multicast features
#             # ----------------------------

#             "distinct_multicast_conversations": (
#                 distinct_multicast_conversations
#             ),

#             "outgoing_multicast_packet_count": (
#                 outgoing_multicast_packet_count
#             ),

#             "outgoing_multicast_total_bytes": (
#                 outgoing_multicast_total_bytes
#             ),

#             # ----------------------------
#             # Metadata
#             # ----------------------------

#             "is_mock_gateway": True,

#             "is_main_gateway": (
#                 gw == main_gateway
#             ),
#         })

#     # ============================================================
#     # Gateway summary dataframe
#     # ============================================================

#     gateway_summary_df = pd.DataFrame(
#         gateway_rows
#     )

#     # ============================================================
#     # Keep original endpoint device summaries
#     #
#     # Remove the real gateway because it is being replaced by
#     # the mock gateways.
#     # ============================================================

#     device_summary = unicast_summary_df[
#         unicast_summary_df["mac"] != gateway_mac
#     ].copy()

#     device_summary[
#         "is_mock_gateway"
#     ] = False

#     device_summary[
#         "is_main_gateway"
#     ] = False

#     # ============================================================
#     # Combine endpoint devices + synthetic gateways
#     # ============================================================

#     result_df = pd.concat(
#         [
#             device_summary,
#             gateway_summary_df,
#         ],
#         ignore_index=True,
#         sort=False,
#     )

#     # ============================================================
#     # Create device -> gateway assignment table
#     # ============================================================

#     assignment_df = pd.DataFrame([
#         {
#             "device_mac": mac,

#             "device_name": (
#                 device_dict[mac]["device_name"]
#             ),

#             "assigned_gateway": gw,

#             "assigned_to_main": (
#                 gw == main_gateway
#             ),
#         }
#         for mac, gw in device_to_gateway.items()
#     ])

#     # ============================================================
#     # Return results
#     # ============================================================

#     return (
#         result_df,
#         gateway_summary_df,
#         assignment_df,
#         main_gateway,
#         peer_traffic,
#     )

In [ ]:
NUMBER_MOCK_GATEWAY = 3

In [ ]:
(
    mock_summary_df,
    gateway_summary_df,
    assignment_df,
    main_gateway,
    peer_traffic,
) = create_mock_gateway_summary(
    device_info=device_info,
    unicast_summary_df=unicast_summary_df,
    device_dict=device_dict_1,
    gateway_mac=GATEWAY_MAC,
    n_gateways=NUMBER_MOCK_GATEWAY,
    random_state=42,
)

In [ ]:
mock_summary_df

In [ ]:
# mock_summary_df["multicast_outgoing_packet_count_per_device"] = (
#     mock_summary_df["outgoing_multicast_packet_count"]
#     .div(mock_summary_df["distinct_multicast_conversations"])
#     .fillna(0)
# )

# mock_summary_df["multicast_outgoing_bytes_per_device"] = (
#     mock_summary_df["outgoing_multicast_total_bytes"]
#     .div(mock_summary_df["distinct_multicast_conversations"])
#     .fillna(0)
# )

# mock_summary_df["multicast_outgoing_bytes_per_packet"] = (
#     mock_summary_df["outgoing_multicast_total_bytes"]
#     .div(mock_summary_df["outgoing_multicast_packet_count"])
#     .fillna(0)
# )
# EXTRA_COLS = ["multicast_outgoing_packet_count_per_device","multicast_outgoing_bytes_per_device","multicast_outgoing_bytes_per_packet"]

In [ ]:
# def feature_selection(all_features: List[str], mock_gate_way_num: int, df: pd.DataFrame) -> List[List[str]]:
#     feature_combinations = [
#         list(combo)
#         for r in range(1, len(all_features) + 1)
#         for combo in combinations(all_features, r)
#     ]
#     valid_combos = []
#     for combo in feature_combinations:
#         features_with_mac = df[
#             feature_cols + ["device_name", "mac"]
#         ]
#         features = df[
#             feature_cols
#         ]
#         scaler = StandardScaler()
#         X_scaled = scaler.fit_transform(features)

#         # kmeans = KMeans(
#         #     n_clusters=2,
#         #     random_state=42,
#         #     n_init=20
#         # )
        
#         # features_with_mac["cluster"] = kmeans.fit_predict(X_scaled)

#         model = AgglomerativeClustering(
#             n_clusters=2
#         )

#         features_with_mac["cluster"] = (
#             model.fit_predict(X_scaled)
#         )

#         # query for all routers
#         router_summary_df = features_with_mac[features_with_mac["device_name"].str.startswith("Mock Gateway", na=False)]
#         router_cluster_id = router_summary_df["cluster"].value_counts().idxmax()

#         total_correctly_classified_router = len(router_summary_df[router_summary_df["cluster"] == router_cluster_id])
#         all_device_classified_as_router = len(features_with_mac[features_with_mac["cluster"] == router_cluster_id])

#         if total_correctly_classified_router == all_device_classified_as_router and total_correctly_classified_router == mock_
#         _num:
#             print(f"{combo}:\n\ttotal_correctly_classified_router: {total_correctly_classified_router}\t\tall_device_classified_as_router: {all_device_classified_as_router}")
#             valid_combos.append(combo)

#     return valid_combos

# valid_feature_combos = feature_selection(feature_cols + EXTRA_COLS, NUMBER_MOCK_GATEWAY, mock_summary_df)

# valid_feature_combos

# # display(
# #     mock_summary_df[
# #         [
# #             "mac",
# #             "device_name",
# #             "cluster"
# #         ] + feature_cols
# #     ].sort_values("cluster")
# # )
# # plotKMeans(mock_summary_df, feature_cols)

- I've tried different configuration and laptop keeps showing up which means we can not only rely on the volume (packet count, packet bytes). There can be something else we should add to differentiate macbook from the gateway.
- Another feature we can add is the number of devices communicating with this device. Usually the router will communicate with more devices. So that might help in the clustering?

# Note on September 15th, 2026
- Add more features/dimensions for the KMeans
- Try to cluster IoT vs non-IoT devices
- Try Kaggle dataset

# Note on September 17th, 2026
-  How many unique devices? Threshold on packet count, packet bytes, packet rate.

# Note
- Trying every possible combination but can not separate it with KMeans
- KMeans still misclassified when we run for 2 mock devices
- Agglomerative is better at 2 devices but can not classify 3 devices
- Normal features doesn't seem to work because in the current dataset, the non-IoT devices are not really active, thus overall traffic volume would kind of be the same as the IoT device. However, one thing we can look at is the packet rate? 

# Now trying IoT and non-IoT

In [ ]:
# from sklearn.mixture import GaussianMixture
# from sklearn.cluster import SpectralClustering
# feature_cols = ["outgoing_packet_count","outgoing_total_bytes","incoming_packet_count","incoming_total_bytes", "total_packet_count", "total_bytes","distinct_multicast_conversations","outgoing_multicast_packet_count","outgoing_multicast_total_bytes"]
# # feature_cols = ["total_packet_count", "total_bytes","distinct_multicast_conversations","outgoing_multicast_total_bytes"]
# unicast_summary_no_gw = unicast_summary_df[unicast_summary_df["mac"] != GATEWAY_MAC]

In [ ]:
# unicast_summary_no_gw["multicast_outgoing_packet_count_per_device"] = (
#     unicast_summary_no_gw["outgoing_multicast_packet_count"]
#     .div(unicast_summary_no_gw["distinct_multicast_conversations"])
#     .fillna(0)
# )

# unicast_summary_no_gw["multicast_outgoing_bytes_per_device"] = (
#     unicast_summary_no_gw["outgoing_multicast_total_bytes"]
#     .div(unicast_summary_no_gw["distinct_multicast_conversations"])
#     .fillna(0)
# )

# unicast_summary_no_gw["multicast_outgoing_bytes_per_packet"] = (
#     unicast_summary_no_gw["outgoing_multicast_total_bytes"]
#     .div(unicast_summary_no_gw["outgoing_multicast_packet_count"])
#     .fillna(0)
# )
# EXTRA_COLS = ["multicast_outgoing_packet_count_per_device","multicast_outgoing_bytes_per_device","multicast_outgoing_bytes_per_packet"]

In [ ]:
# unicast_summary_no_gw

In [ ]:
# features = unicast_summary_no_gw[EXTRA_COLS + feature_cols]

In [ ]:


# # features = unicast_summary_df[
# #     ["total_packet_count"]
# # ]

# # X = np.log1p(features)
# scaler = StandardScaler()
# X_scaled = scaler.fit_transform(features)

# # kmeans = KMeans(
# #     n_clusters=2,
# #     random_state=42,
# #     n_init=20
# # )
# # unicast_summary_no_gw["cluster"] = kmeans.fit_predict(X_scaled)

# # dbscan = DBSCAN(
# #     eps=1.5,
# #     min_samples=3
# # )

# # unicast_summary_no_gw["cluster"] = dbscan.fit_predict(X_scaled)

# model = AgglomerativeClustering(
#     n_clusters=2
# )

# unicast_summary_no_gw["cluster"] = (
#     model.fit_predict(X_scaled)
# )

# # model = GaussianMixture(
# #     n_components=2,
# #     random_state=42
# # )

# # unicast_summary_no_gw["cluster"] = (
# #     model.fit_predict(X_scaled)
# # )

# # model = SpectralClustering(
# #     n_clusters=2,
# #     affinity="nearest_neighbors",
# #     random_state=42
# # )

# # unicast_summary_no_gw["cluster"] = (
# #     model.fit_predict(X_scaled)
# # )

# display(
#     unicast_summary_no_gw[
#         [
#             "mac",
#             "device_name",
#             "cluster"
#         ] + feature_cols
#     ].sort_values("cluster")
# )

In [ ]:
# plotKMeans(unicast_summary_no_gw, feature_cols)